# Phase 1 — Data Quality & Preparation
**Group 7 | Housewares (`utilidades_domesticas`) | Supply Chain Director Persona**
**Dataset:** Olist Brazilian E-Commerce (2016-2018)

In this phase we:
1. Filter all master tables down to Housewares orders only.
2. Audit the filtered dataset for 5 distinct data quality issues.
3. Document each issue, its likely cause, and the resolution applied.

---


### Step 1: Load Libraries & Raw Data

In [4]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

products    = pd.read_csv('data/olist_products_dataset.csv')
order_items = pd.read_csv('data/olist_order_items_dataset.csv')
orders      = pd.read_csv('data/olist_orders_dataset.csv')
customers   = pd.read_csv('data/olist_customers_dataset.csv')
reviews     = pd.read_csv('data/olist_order_reviews_dataset.csv')
payments    = pd.read_csv('data/olist_order_payments_dataset.csv')

print("All datasets loaded successfully.")
print(f"  products:    {products.shape}")
print(f"  order_items: {order_items.shape}")
print(f"  orders:      {orders.shape}")
print(f"  customers:   {customers.shape}")
print(f"  reviews:     {reviews.shape}")
print(f"  payments:    {payments.shape}")


All datasets loaded successfully.
  products:    (32951, 9)
  order_items: (112650, 7)
  orders:      (99441, 8)
  customers:   (99441, 5)
  reviews:     (99224, 7)
  payments:    (103886, 5)


### Step 2: Scope Data to Housewares (`utilidades_domesticas`)

In [5]:
hw_products  = products[products['product_category_name'] == 'utilidades_domesticas'].copy()
hw_items     = order_items[order_items['product_id'].isin(hw_products['product_id'])].copy()
hw_order_ids = hw_items['order_id'].unique()
hw_orders    = orders[orders['order_id'].isin(hw_order_ids)].copy()
hw_customers = customers[customers['customer_id'].isin(hw_orders['customer_id'])].copy()
hw_reviews   = reviews[reviews['order_id'].isin(hw_order_ids)].copy()
hw_payments  = payments[payments['order_id'].isin(hw_order_ids)].copy()

print("=== Housewares Subset Sizes ===")
print(f"  Distinct Products : {len(hw_products):,}")
print(f"  Order Item Rows   : {len(hw_items):,}")
print(f"  Unique Orders     : {len(hw_orders):,}")
print(f"  Customers         : {len(hw_customers):,}")
print(f"  Reviews           : {len(hw_reviews):,}")
print(f"  Payment Rows      : {len(hw_payments):,}")


=== Housewares Subset Sizes ===
  Distinct Products : 2,335
  Order Item Rows   : 6,964
  Unique Orders     : 5,884
  Customers         : 5,884
  Reviews           : 5,865
  Payment Rows      : 6,240


### Step 3: Data Quality Checks

#### Issue 1 — Missing Delivery Timestamps
Fields: `order_delivered_customer_date` and `order_delivered_carrier_date`


In [6]:
missing_orders = hw_orders[[
    'order_status',
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]].isnull().sum().rename('Null Count')

print("Missing values in Orders table:")
print(missing_orders.to_string())
print("\nOrders breakdown by status:")
print(hw_orders['order_status'].value_counts().to_string())


Missing values in Orders table:
order_status                       0
order_purchase_timestamp           0
order_approved_at                  0
order_delivered_carrier_date      61
order_delivered_customer_date    141
order_estimated_delivery_date      0

Orders breakdown by status:
delivered     5743
shipped         76
canceled        37
processing      17
invoiced        10
approved         1


#### Issue 2 — Chronological Timestamp Inversions

In [7]:
for col in ['order_purchase_timestamp', 'order_approved_at',
            'order_delivered_carrier_date', 'order_delivered_customer_date',
            'order_estimated_delivery_date']:
    hw_orders[col] = pd.to_datetime(hw_orders[col])

delivered_only  = hw_orders[hw_orders['order_status'] == 'delivered'].copy()
invalid_carrier = (delivered_only['order_purchase_timestamp'] > delivered_only['order_delivered_carrier_date']).sum()
invalid_deliver = (delivered_only['order_delivered_carrier_date'] > delivered_only['order_delivered_customer_date']).sum()
late_count      = (delivered_only['order_delivered_customer_date'] > delivered_only['order_estimated_delivery_date']).sum()
total_delivered = len(delivered_only)

print(f"Anomaly: Purchase recorded AFTER carrier handoff  : {invalid_carrier} orders")
print(f"Anomaly: Carrier handoff recorded AFTER delivery  : {invalid_deliver} orders")
print(f"\nLate deliveries (actual > estimated)            : {late_count} out of {total_delivered}")
print(f"On-time delivery rate (delivered orders)         : {round((1 - late_count/total_delivered)*100, 2)}%")


Anomaly: Purchase recorded AFTER carrier handoff  : 9 orders
Anomaly: Carrier handoff recorded AFTER delivery  : 3 orders

Late deliveries (actual > estimated)            : 399 out of 5743
On-time delivery rate (delivered orders)         : 93.05%


#### Issue 3 — Missing Review Comment Text (`review_comment_message`)

In [8]:
missing_reviews = hw_reviews[['review_score', 'review_comment_message']].isnull().sum().rename('Null Count')
print("Missing values in Reviews table:")
print(missing_reviews.to_string())
print(f"\n-> {hw_reviews['review_comment_message'].isnull().mean()*100:.1f}% of reviews have no written comment.")
print("-> review_score (0 nulls) is used for satisfaction analysis instead.")


Missing values in Reviews table:
review_score                 0
review_comment_message    3497

-> 59.6% of reviews have no written comment.
-> review_score (0 nulls) is used for satisfaction analysis instead.


#### Issue 4 — Orders With No Review (Orphan Orders)

In [9]:
orders_without_review = set(hw_order_ids) - set(hw_reviews['order_id'])
print(f"Houseware orders with no review entry : {len(orders_without_review)}")
print(f"As % of total Houseware orders        : {round(len(orders_without_review)/len(hw_order_ids)*100, 2)}%")
print("\n-> Resolution: LEFT JOIN for operations; INNER JOIN when correlating delay with rating.")


Houseware orders with no review entry : 41
As % of total Houseware orders        : 0.7%

-> Resolution: LEFT JOIN for operations; INNER JOIN when correlating delay with rating.


#### Issue 5 — Dual Customer Identifier (Repeat Buyers Masked)

In [10]:
dup_cust_id  = hw_customers['customer_id'].duplicated().sum()
repeat_uid   = hw_customers['customer_unique_id'].duplicated().sum()
total_unique = hw_customers['customer_unique_id'].nunique()

print(f"Duplicate customer_id (per-order key)     : {dup_cust_id}")
print(f"Repeat buyers via customer_unique_id       : {repeat_uid}")
print(f"Truly distinct Houseware customers         : {total_unique:,}")
print("\n-> Resolution: Group by customer_unique_id for any retention analysis.")


Duplicate customer_id (per-order key)     : 0
Repeat buyers via customer_unique_id       : 63
Truly distinct Houseware customers         : 5,821

-> Resolution: Group by customer_unique_id for any retention analysis.


---
### Phase 1 Summary — Data Quality Log

| # | Issue Found | Table / Field | Likely Cause | Resolution Applied |
|---|---|---|---|---|
| 1 | **141 missing customer delivery dates** | `orders` / `order_delivered_customer_date` | Orders still in-transit, cancelled, or processing | Filter to `order_status == 'delivered'` before computing KPIs |
| 2 | **12 timestamp inversions** (9 purchase>carrier, 3 carrier>delivery) | `orders` / timestamp columns | Timezone errors or courier scan sequence issues | Exclude these 12 records from lead-time calculations |
| 3 | **3,497 missing review comments** (59.2% of reviews) | `order_reviews` / `review_comment_message` | Written comments are optional in Olist surveys | Use numeric `review_score` (0 nulls) exclusively |
| 4 | **41 orders with no review** | `orders` vs `order_reviews` | Customer opted out of review prompt | LEFT JOIN for operations; INNER JOIN for delay-rating correlation |
| 5 | **63 repeat buyer sessions** masked as new customers | `customers` / `customer_id` vs `customer_unique_id` | New `customer_id` created per order session by Olist | Group by `customer_unique_id` for retention analysis |
